In [56]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchattacks
from torchattacks import FGSM, PGD, MIFGSM 
import pandas as pd
from functions import encode_variable
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, recall_score
import torch.nn.functional as F
from pathlib import Path

In [57]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchattacks
from torchattacks import FGSM, PGD, MIFGSM 
import pandas as pd
from functions import encode_variable
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, recall_score
import torch.nn.functional as F
from pathlib import Path

In [58]:
class RealTabularMIFGSM(torchattacks.MIFGSM):
    def __init__(self, model, eps=0.03, alpha=0.007, steps=10, decay=1.0, mask=None, feature_min = None, feature_max = None):
        super().__init__(model, eps, alpha, steps, decay)
        self.loss = nn.CrossEntropyLoss()  
        self.mask = mask 
        self.feature_min = feature_min
        self.feature_max = feature_max
        
    def forward(self, images, labels):
        """Overrides MIFGSM for tabular data."""
        
        adv_images = images.clone().detach()
        adv_images.requires_grad = True 
        
        momentum = 0
        for _ in range(self.steps):
            outputs = self.model(adv_images)
            loss = self.loss(outputs, labels)

            grad = torch.autograd.grad(loss, adv_images, retain_graph=True, create_graph=True, allow_unused=True)[0]

            if grad is None:
                print("ERROR: Gradient is None! adv_images is not in the graph!")
                break  

            grad = grad / torch.mean(torch.abs(grad), dim=(1,), keepdim=True)
            grad = grad + momentum * self.decay
            momentum = grad

            perturbation = self.alpha * grad.sign()
            if self.mask is not None:
                perturbation = perturbation * self.mask  # Zero out protected features

            adv_images = adv_images + perturbation
            adv_images = torch.clamp(adv_images, min=self.feature_min, max=self.feature_max) 
    
        return adv_images

In [59]:
# Generating, training and testing surrogate model
class SurrogateModel(nn.Module):
    def __init__(self, input_dim):
        super(SurrogateModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128) 
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 2)  # Binary classification (Normal vs. Anomalous)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

def test_surrogate(model_surrogate, X_test, y_test, device):
    model_surrogate.eval()
    X_test, y_test = X_test.to(device), y_test.to(device)

    with torch.no_grad():
        y_pred = model_surrogate(X_test).argmax(axis=1).cpu().numpy()

    accuracy = accuracy_score(y_test.cpu().numpy(), y_pred)
    recall = recall_score(y_test.cpu().numpy(), y_pred)
    f1 = f1_score(y_test.cpu().numpy(), y_pred, average='weighted')

    results = {
        "Accuracy": accuracy,
        "Recall": recall,
        "F1 Score": f1
    }
    print(f"Surrogate Model Results: {results}")
    return results

def train_surrogate(X_train, y_train, device):
    input_dim = X_train.shape[1]
    model_surrogate = SurrogateModel(input_dim).to(device)
    optimizer = optim.Adam(model_surrogate.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)

    for epoch in range(8):  # epochs
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model_surrogate(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

    print("Surrogate model training complete.")
    return model_surrogate

In [81]:
from sklearn.preprocessing import StandardScaler
def process_data(input_data, device):
    data = input_data.dropna(axis=0)
    print(data)
    #print(data.nunique() > 1)
    #data = data.loc[:, data.nunique() > 1]
    data.columns = [col.strip().lower() for col in data.columns]
    new_cols = ['timestamp' if 'starttimestamp' in col else col for col in data.columns]
    data.columns = new_cols
    try:
        data['timestamp'] = pd.to_datetime(data['timestamp'], errors='coerce')
        data = data.sort_values(by='timestamp', ascending=True)
    except:
        print("timestamp fallido")
    features = [col for col in data.columns if col not in ['city', 'timestamp', 'anomaly', 'normal/attack',  'startdate', 'weekdaystart', 'yearstart', 'hourstart', 'minutestart', 'enddate', 'endtimestamp', 'weekdayend', 'yearend', 'hourend', 'minuteend', 'class']]
    if "connectortype" in [col for col in data.columns]:
        features = ['connectortype', 'durationcharge' , 'durationsession' , 'energy' , 'tariff' ,
                'cost' , 'meanpower', 'maxpower']
        data = encode_variable(data, data.columns.get_loc("connectortype"))

    for feature in features:
        data[feature] = pd.to_numeric(data[feature], errors='coerce')

    data.dropna()
    #data = data.sample(frac=1).reset_index(drop=True)
    x_data = data.loc[:, features].select_dtypes(include=[np.number])
    y_data = data.loc[:, ['anomaly']].values.ravel()
    print(y_data)
    # Now scale
    scaler = StandardScaler()
    x_data = scaler.fit_transform(x_data)

    size = len(x_data)
    size_init = int(size * 0.8)
    x_train = x_data[:size_init]
    y_train = y_data[:size_init]
    x_test = x_data[size_init:]
    y_test = y_data[size_init:]

    #x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.3, random_state=42, stratify=y_data)

    df = pd.DataFrame(x_test, columns=features)
    df["anomaly"] = y_test
    df = df.sample(frac=1).reset_index(drop=True)
    x_test = df.loc[:, features].values
    y_test = df.loc[:, ['anomaly']].values.ravel()
    x_train = np.array(x_train, dtype=np.float32) 
    x_test = np.array(x_test, dtype=np.float32)
    x_train = torch.tensor(x_train, dtype=torch.float32).to(device)
    y_train = torch.tensor(y_train, dtype=torch.long).to(device)
    x_test = torch.tensor(x_test, dtype=torch.float32).to(device)
    y_test = torch.tensor(y_test, dtype=torch.long).to(device)

    return data, x_train, x_test, y_train, y_test, features
			
def load_anomalous_test_samples(x_test, y_test):
    anomalous_indices = np.where(y_test == 1)[0] 
    X_anomalous = x_test[anomalous_indices]
    y_anomalous = y_test[anomalous_indices]
    return X_anomalous, y_anomalous, anomalous_indices

def process_anomalous_samples(x_train, y_train, x_test, y_test, device):
    # Prepare anomalous x y for perturbation
    X_anomalous_test, y_anomalous_test, anomalous_indices_test = load_anomalous_test_samples(x_test, y_test)
    X_anomalous_train, y_anomalous_train, anomalous_indices_train = load_anomalous_test_samples(x_train, y_train)
    x_anom_train = torch.tensor(X_anomalous_train, dtype=torch.float32).clone().detach().requires_grad_(True).to(device)
    x_anom_train = torch.tensor(X_anomalous_train, dtype=torch.float32).to(device)
    x_anom_train.requires_grad = True
    y_anom_train = torch.tensor(y_anomalous_train, dtype=torch.long).to(device)
    y_anom_train = y_anom_train.view(-1).to(device)
    
    x_anom_test = torch.tensor(X_anomalous_test, dtype=torch.float32).clone().detach().requires_grad_(True).to(device)
    x_anom_test = torch.tensor(X_anomalous_test, dtype=torch.float32).to(device)
    x_anom_test.requires_grad = True
    y_anom_test = torch.tensor(y_anomalous_test, dtype=torch.long).to(device)
    y_anom_test = y_anom_test.view(-1).to(device)

    return x_anom_train, y_anom_train, x_anom_test, y_anom_test, anomalous_indices_test, anomalous_indices_train

def extract_column_stats(df):
    stats = {}
    feature_min = []
    feature_max = []
    for column in df.columns:
        stats[column] = {
            'max': df[column].max(),
            'min': df[column].min(),
            'mean': df[column].mean()
        }
        feature_min.append(df[column].min())
        feature_max.append(df[column].max())
    print(stats)
    print(feature_min)
    print(feature_max)
    return stats, feature_min, feature_max

def generateMask(x): 
    mask = []
    for i in range(x.cpu().numpy().shape[1]):  
        col = x.cpu().numpy()[:, i] 
        unique_values = np.unique(col)  
        if len(unique_values) < 5 or np.issubdtype(col.dtype, np.integer):
            mask.append(0) 
        else:
            mask.append(1)
    print(mask)
    return mask

def save_samples_csv(adversarial_samples, adv_dir, data, features, y):
    adversarial_dataset = pd.DataFrame(adversarial_samples.detach().cpu().numpy(), columns=data.loc[:, features].columns)
    adversarial_dataset['anomaly'] = y.detach().cpu().numpy()
    adversarial_dataset.to_csv(adv_dir, index=False)
    #print(f"{adv_dir} converted to CSV")

def performAttacks(steps, x, y, mask, feature_min, feature_max, model_surrogate, file, stage, data, features, eps, alpha):
    mifgsm = RealTabularMIFGSM(model_surrogate, eps=eps, alpha=alpha, steps=steps, decay=0.8, mask=mask, feature_min=feature_min,
                                    feature_max=feature_max)
    mifgsm_samples = mifgsm(x, y)
    adv_dir = f'./data/aexamples/{file}/mifgsm/{stage}/adversarialexamples_eps{str(eps)}.csv'
    Path(adv_dir).parent.mkdir(parents=True, exist_ok=True)
    save_samples_csv(mifgsm_samples, adv_dir, data, features, y)

def attack(x, y, mask, feature_min, feature_max, model_surrogate, file, stage, data, features):
    steps = 20
    epsilons = [0.0]
     
    for value in np.arange(0.003, 0.031, 0.006):
        eps = round(value, 3)
        alpha = round(eps/steps, 4)
        performAttacks(steps, x, y, mask, feature_min, feature_max, model_surrogate, file, stage, data, features, eps, alpha)
        epsilons.append(eps)
    for value in np.arange(0.03, 0.3, 0.06):
        eps = round(value, 3)
        alpha = round(eps/steps, 4)
        performAttacks(steps, x, y, mask, feature_min, feature_max, model_surrogate, file, stage, data, features, eps, alpha)
        epsilons.append(eps)
    if "train" in stage:
        for value in np.arange(0.003, 0.3, 0.05):
            eps = round(value, 3)
            alpha = round(eps/steps, 4)
            performAttacks(steps, x, y, mask, feature_min, feature_max, model_surrogate, file, stage, data, features, eps, alpha)

    return epsilons


In [85]:
def generate_ae_examples():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}')
    data_path = Path('./data')
    original_results = {}
    files = [f for f in data_path.iterdir() if f.is_file()]
    for file in files:
        file_name = file.with_suffix('').as_posix().split('/')[1]
        original_results[file_name] = {}
        print(f"Processing file {file_name}")
        if 'xlsx' in file.name:
            data = pd.read_excel(file)
        else:
            data = pd.read_csv(file)
        data, x_train, x_test, y_train, y_test, features = process_data(data, device)
        print(np.unique(y_train.cpu().numpy(), return_counts=True))
        print(np.unique(y_test.cpu().numpy(), return_counts=True))
        # Train and test surrogate model
        model_surrogate = train_surrogate(x_train, y_train, device)
        original_results[file_name]['results'] = test_surrogate(model_surrogate, x_test, y_test, device)
        original_results[file_name]['x_test'] = x_test
        original_results[file_name]['y_test'] = y_test
        original_results[file_name]['x_train'] = x_train
        original_results[file_name]['y_train'] = y_train
        original_results[file_name]['features'] = features
        original_results[file_name]['model_surrogate'] = model_surrogate 
        # Generate data constraints
        stats, feature_min, feature_max = extract_column_stats(data.loc[:, features])
        mask = generateMask(x_train)
        feature_min = torch.tensor(np.array(feature_min), dtype=torch.float32).to(device)
        feature_max = torch.tensor(np.array(feature_max), dtype=torch.float32).to(device)  
        mask = torch.tensor(np.array(mask), dtype=torch.float32).to(device) 

        x_anom_train, y_anom_train, x_anom_test, y_anom_test, anomalous_indices_test, anomalous_indices_train = process_anomalous_samples(x_train.cpu(), y_train.cpu(), x_test.cpu(), y_test.cpu(), device)
        original_results[file_name]['anomalous_indices_test'] = anomalous_indices_test
        original_results[file_name]['anomalous_indices_train'] = anomalous_indices_train
        epsilons = attack(x_anom_train, y_anom_train, mask, feature_min, feature_max, model_surrogate, file_name, 'train', data, features)
        epsilons = attack(x_anom_test, y_anom_test, mask, feature_min, feature_max, model_surrogate, file_name, 'test', data, features)

    return original_results, epsilons, device
        
original_results, epsilons, device = generate_ae_examples()

Device: cuda
Processing file Dundee_Clean_With_Anomalies_All
       country    city connectorType  durationCharge  durationSession  \
0           UK  Dundee         Rapid            0.16             0.17   
1           UK  Dundee         Rapid            0.43             0.43   
2           UK  Dundee         Rapid            0.12             0.12   
3           UK  Dundee         Rapid            0.17             0.17   
4           UK  Dundee         Rapid            0.64             0.65   
...        ...     ...           ...             ...              ...   
210509      UK  Dundee         Rapid            0.12             0.12   
210510      UK  Dundee   Ultra-Rapid            1.47             1.47   
210511      UK  Dundee         Rapid            0.63             0.63   
210512      UK  Dundee          Fast            0.85             0.85   
210513      UK  Dundee         Rapid            1.02             3.04   

         energy    tariff     cost  meanPower  maxPower  ... h

In [86]:
# Test examples
def read_adv_data(advPath):
    adversarial_data = pd.read_csv(advPath)
    features = [col for col in adversarial_data.columns if col not in ['timestamp', 'anomaly', 'normal/attack',  'startdate', 'weekdaystart', 'yearstart', 'hourstart', 'minutestart', 'enddate', 'endtimestamp', 'weekdayend', 'yearend', 'hourend', 'minuteend', 'class']]
    adversarial_x_data = adversarial_data.loc[:, features].values
    adversarial_y_data = adversarial_data.loc[:, ['anomaly']].values.ravel()
    return adversarial_x_data, adversarial_y_data, features
    
def test_adversarial_samples(advPath, x_test, y_test, model_surrogate, anomalous_indices, device):
    adversarial_x_data, adversarial_y_data, _ = read_adv_data(advPath)
    X_modified, Y_modified  = x_test.clone().cpu().numpy(), y_test.clone().cpu().numpy()
    X_modified[anomalous_indices] = adversarial_x_data
    Y_modified[anomalous_indices] = adversarial_y_data

    try:
        y_pred = model_surrogate.predict(X_modified)
    except: 
        model_surrogate.eval()
        tensor_X_modified = torch.tensor(np.array(X_modified, dtype=np.float32), dtype=torch.float32).to(device)
        with torch.no_grad():
            y_pred = model_surrogate(tensor_X_modified).argmax(axis=1).cpu().numpy()

    accuracy = accuracy_score(Y_modified, y_pred)
    recall = recall_score(Y_modified, y_pred)
    f1 = f1_score(Y_modified, y_pred)
    return  accuracy, f1, recall

def initializeResults(original_results, file_name, epsilons):
    metrics = ["Accuracy", "F1 Score", "Recall", "EIR"]
    attacks = ["MIFGSM"]
    multi_index = pd.MultiIndex.from_product([attacks, metrics], names=["Attack", "Metric"])
    df_results = pd.DataFrame(index=multi_index, columns=epsilons)
    df_results.loc[("MIFGSM", "Accuracy"), 0.0] = original_results[file_name]['results']["Accuracy"]
    df_results.loc[("MIFGSM", "F1 Score"), 0.0] = original_results[file_name]['results']["F1 Score"]
    df_results.loc[("MIFGSM", "Recall"), 0.0] = original_results[file_name]['results']["Recall"]
    df_results.loc[("MIFGSM", "EIR"), 0.0] = "-"
    return df_results

def obtain_adv_results(df_results, original_results, file, stage,  x_test, y_test, anomalous_indices, eps, model_surrogate, device):
    # MIFGSM
    adv_dir = f'./data/aexamples/{file}/mifgsm/{stage}/adversarialexamples_eps{str(eps)}.csv'
    accuracy, f1, recall = test_adversarial_samples(adv_dir, x_test, y_test, model_surrogate, anomalous_indices, device)
    df_results[file].loc[("MIFGSM", "Accuracy"), eps] = accuracy
    df_results[file].loc[("MIFGSM", "F1 Score"), eps] = f1
    df_results[file].loc[("MIFGSM", "Recall"), eps] = recall
    eir = 1 - (df_results[file].loc[("MIFGSM", "Recall"), eps] / max(original_results[file]['results']["Recall"], 1e-8))
    df_results[file].loc[("MIFGSM", "EIR"), eps] = eir * 100

    return df_results

def completeTest(original_results, epsilons, device):
    df_results = {}
    stage = 'test'
    
    for file in original_results:
        df_results[file] = initializeResults(original_results, file, epsilons)
        x_test, y_test = original_results[file]['x_test'], original_results[file]['y_test']
        anomalous_indices = original_results[file]['anomalous_indices_test']
        model_surrogate = original_results[file]['model_surrogate']
        #for value in np.arange(0.003, 0.031, 0.006): 
        #    eps = round(value, 3)
        #    df_results = obtain_adv_results(df_results, file, stage,  x_test, y_test, anomalous_indices, eps, model_surrogate, device)
        
        #for value in np.arange(0.03, 0.3, 0.06):
        for value in np.arange(0.003, 0.031, 0.006): 
            eps = round(value, 3)
            df_results = obtain_adv_results(df_results, original_results, file, stage,  x_test, y_test, anomalous_indices, eps, model_surrogate, device)

        for value in np.arange(0.03, 0.3, 0.06):
            eps = round(value, 3)
            df_results = obtain_adv_results(df_results, original_results, file, stage,  x_test, y_test, anomalous_indices, eps, model_surrogate, device)

    return df_results
df_results = completeTest(original_results, epsilons, device)

In [84]:
import pickle
with open("df_results.pkl", "wb") as f:
    pickle.dump(df_results, f)
with open("original_results.pkl", "wb") as f:
    pickle.dump(original_results, f)

In [12]:
import pickle
with open("df_results.pkl", "rb") as f:
    df_results = pickle.load(f)
with open("original_results.pkl", "rb") as f:
    original_results = pickle.load(f)

In [87]:
df_results['Dundee_Clean_With_Anomalies_All']

0.000     0.003      0.009      0.015      0.021  \
Attack Metric                                                          
MIFGSM Accuracy  0.816735  0.800869   0.760635   0.708619   0.690426   
       F1 Score  0.810589   0.71483   0.636252   0.519204   0.473374   
       Recall    0.645937  0.607329   0.509421   0.382846   0.338574   
       EIR              -  5.977094  21.134574  40.730136  47.584109   

                     0.027      0.030      0.090      0.150      0.210  \
Attack Metric                                                            
MIFGSM Accuracy   0.681662   0.678906   0.636772   0.631784   0.630193   
       F1 Score   0.450269    0.44286   0.320039   0.304205   0.299091   
       Recall     0.317247   0.310542   0.208011   0.195873   0.192001   
       EIR       50.885827  51.923765  67.797065  69.676092  70.275591   

                     0.270  
Attack Metric               
MIFGSM Accuracy   0.660428  
       F1 Score    0.39128  
       Recall     0.265576  
       EIR       58.885111

In [88]:
df_results['Dundee_Clean_With_Anomalies']

0.000      0.003      0.009      0.015      0.021  \
Attack Metric                                                           
MIFGSM Accuracy  0.818167   0.713463   0.711869   0.709855   0.707953   
       F1 Score  0.800156   0.215947   0.208132   0.198161   0.188641   
       Recall    0.476873   0.130527   0.125254   0.118594   0.112303   
       EIR              -  72.628516  73.734239  75.130941  76.450048   

                     0.027      0.030     0.090      0.150      0.210  \
Attack Metric                                                           
MIFGSM Accuracy   0.706192   0.705604  0.703255   0.702528   0.702109   
       F1 Score   0.179731   0.176742  0.164686   0.160921   0.158743   
       Recall     0.106475   0.104533  0.096762   0.094357   0.092969   
       EIR       77.672163  78.079534  79.70902  80.213385  80.504365   

                     0.270  
Attack Metric               
MIFGSM Accuracy   0.701745  
       F1 Score    0.15685  
       Recall     0.091767  
       EIR       80.756547

In [89]:
df_results['Boulder_Clean_With_Anomalies']

0.000     0.003     0.009     0.015     0.021     0.027  \
Attack Metric                                                                 
MIFGSM Accuracy  0.993241   0.99896   0.99896   0.99896   0.99896   0.99896   
       F1 Score  0.993197  0.996578  0.996578  0.996578  0.996578  0.996578   
       Recall    0.962232       1.0       1.0       1.0       1.0       1.0   
       EIR              - -3.925067 -3.925067 -3.925067 -3.925067 -3.925067   

                    0.030     0.090     0.150     0.210     0.270  
Attack Metric                                                      
MIFGSM Accuracy   0.99896   0.99896   0.99896   0.99896   0.99883  
       F1 Score  0.996578  0.996578  0.996578  0.996578  0.996149  
       Recall         1.0       1.0       1.0       1.0  0.999142  
       EIR      -3.925067 -3.925067 -3.925067 -3.925067 -3.835861

In [90]:
df_results['PaloAlto_2018_2022_Clean_With_Anomalies']

0.000     0.003     0.009     0.015     0.021     0.027  \
Attack Metric                                                                 
MIFGSM Accuracy   0.98055  0.991079  0.991041  0.990849  0.990734  0.990658   
       F1 Score  0.980344  0.971351  0.971225  0.970592  0.970212  0.969958   
       Recall    0.911911  0.980149  0.979901   0.97866  0.977916  0.977419   
       EIR              - -7.482993 -7.455782 -7.319728 -7.238095 -7.183673   

                    0.030     0.090     0.150     0.210     0.270  
Attack Metric                                                      
MIFGSM Accuracy  0.990619  0.988705  0.987748   0.98614  0.984149  
       F1 Score  0.969831  0.963449  0.960229  0.954773  0.947938  
       Recall    0.977171  0.964764  0.958561  0.948139  0.935236  
       EIR      -7.156463 -5.795918 -5.115646 -3.972789 -2.557823

In [91]:
df_results['Perth_Clean_With_Anomalies']

0.000      0.003      0.009      0.015      0.021  \
Attack Metric                                                           
MIFGSM Accuracy  0.987128   0.927866    0.92742   0.925827    0.92519   
       F1 Score  0.986972   0.707946   0.705609    0.69719   0.693792   
       Recall    0.930477   0.554568   0.551738   0.541633   0.537591   
       EIR              -  40.399652  40.703736  41.789748  42.224153   

                     0.027      0.030      0.090      0.150      0.210  \
Attack Metric                                                            
MIFGSM Accuracy   0.924489    0.92417    0.91442   0.904034   0.895622   
       F1 Score   0.690034   0.688318   0.633561    0.56996   0.513947   
       Recall     0.533145   0.531124   0.469281   0.403395    0.35004   
       EIR       42.701998  42.919201  49.565595  56.646394  62.380539   

                     0.270  
Attack Metric               
MIFGSM Accuracy   0.886128  
       F1 Score   0.445203  
       Recall     0.289814  
       EIR       68.853171

In [92]:
df_results['Netherlands_Clean_With_Anomalies']

0.000      0.003      0.009     0.015     0.021     0.027  \
Attack Metric                                                                   
MIFGSM Accuracy  0.918755   0.935732   0.933306  0.929264  0.925627  0.921989   
       F1 Score  0.912109   0.824309   0.816463   0.80315  0.790909  0.778416   
       Recall    0.635317   0.715931   0.704415  0.685221  0.667946  0.650672   
       EIR              - -12.688822 -10.876133 -7.854985 -5.135952 -2.416918   

                    0.030      0.090      0.150     0.210      0.270  
Attack Metric                                                         
MIFGSM Accuracy   0.92118   0.899353   0.883185   0.86823   0.854891  
       F1 Score  0.775604   0.694479   0.627097  0.558266    0.49078  
       Recall    0.646833   0.543186   0.466411  0.395393   0.332054  
       EIR      -1.812689  14.501511  26.586103  37.76435  47.734139

In [61]:
df_results['Canada1_clean_WithAnomalies']

0.000     0.003     0.009     0.015     0.021     0.027  \
Attack Metric                                                                 
MIFGSM Accuracy  0.978071  0.977443  0.976814  0.975414  0.974186  0.972986   
       F1 Score  0.978104  0.943621  0.941959  0.938238  0.934951  0.931721   
       Recall    0.949034  0.945884  0.942734  0.935719  0.929563   0.92355   
       EIR              -  0.331875   0.66375  1.402927  2.051591  2.685171   

                    0.030     0.090      0.150      0.210      0.270  
Attack Metric                                                         
MIFGSM Accuracy  0.972671  0.961843   0.949886     0.9399   0.933171  
       F1 Score  0.930871  0.900762    0.86548   0.834233   0.812189  
       Recall    0.921976  0.867717   0.807802   0.757767   0.724052  
       EIR       2.851109  8.568412  14.881581  20.153869  23.706441

In [62]:
df_results['Germany_clean_WithAnomalies']

0.000     0.003     0.009     0.015     0.021     0.027  \
Attack Metric                                                                 
MIFGSM Accuracy  0.893157  0.892243  0.891471  0.889986  0.888714  0.887543   
       F1 Score  0.886156  0.684591  0.681614  0.675843  0.670864  0.666243   
       Recall    0.590551   0.58597  0.582105   0.57466  0.568289  0.562419   
       EIR              -  0.775758  1.430303  2.690909  3.769697  4.763636   

                    0.030      0.090      0.150      0.210      0.270  
Attack Metric                                                          
MIFGSM Accuracy  0.887214      0.876   0.865529   0.856057   0.848271  
       F1 Score  0.664941   0.618931    0.57301   0.528763   0.490282  
       Recall    0.560773   0.504581   0.452112   0.404653   0.365641  
       EIR       5.042424  14.557576  23.442424  31.478788  38.084848

In [63]:
df_results['Portugal_clean_WithAnomalies']

0.000     0.003     0.009     0.015     0.021     0.027  \
Attack Metric                                                                 
MIFGSM Accuracy  0.925257  0.790586  0.790586  0.790586  0.790586  0.790586   
       F1 Score   0.92052       0.0       0.0       0.0       0.0       0.0   
       Recall    0.674803       0.0       0.0       0.0       0.0       0.0   
       EIR              -     100.0     100.0     100.0     100.0     100.0   

                    0.030     0.090     0.150     0.210     0.270  
Attack Metric                                                      
MIFGSM Accuracy  0.790586  0.790586  0.790586  0.790586  0.790586  
       F1 Score       0.0       0.0       0.0       0.0       0.0  
       Recall         0.0       0.0       0.0       0.0       0.0  
       EIR          100.0     100.0     100.0     100.0     100.0

In [8]:
df_results['US_Alabama_clean_WithAnomalies']

0.000     0.003     0.009     0.015     0.021     0.027  \
Attack Metric                                                                 
MIFGSM Accuracy    0.9385  0.937557  0.936586  0.934971  0.933957  0.932686   
       F1 Score  0.937374  0.835658  0.832674   0.82768  0.824521  0.820536   
       Recall    0.800215   0.79549  0.790623  0.782534  0.777452  0.771081   
       EIR              -  0.590393  1.198676    2.2095  2.844619  3.640755   

                    0.030      0.090      0.150      0.210      0.270  
Attack Metric                                                          
MIFGSM Accuracy  0.932343   0.921771   0.911486   0.901543   0.891571  
       F1 Score  0.819457   0.785188   0.749879    0.71374   0.675336  
       Recall    0.769363   0.716392   0.664853   0.615032   0.565068  
       EIR       3.855443  10.474998  16.915645  23.141605  29.385455

In [65]:
df_results['vancover_clean_WithAnomalies']

0.000     0.003     0.009     0.015     0.021     0.027  \
Attack Metric                                                                 
MIFGSM Accuracy  0.880829  0.879914  0.879143  0.877743  0.876743    0.8758   
       F1 Score  0.869186  0.625368  0.622051  0.615992  0.611631  0.607494   
       Recall      0.5068  0.502219  0.498354  0.491339  0.486328  0.481603   
       EIR              -  0.903955  1.666667  3.050847  4.039548  4.971751   

                    0.030      0.090      0.150      0.210      0.270  
Attack Metric                                                          
MIFGSM Accuracy  0.875414   0.865171   0.857129     0.8504   0.843957  
       F1 Score  0.605795   0.559096   0.520175   0.485961   0.451684  
       Recall    0.479671   0.428346   0.388046   0.354331   0.322047  
       EIR       5.353107  15.480226  23.432203  30.084746  36.454802

In [93]:
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

def trainModelsTransfer(x_train, y_train):
    mask = ~torch.isnan(x_train).any(dim=1)
    x_train = x_train[mask]
    y_train = y_train[mask]
    models = []
    # 1. - Train Catboost   
    print("[CATBOOST]")     
    catboost_model = CatBoostClassifier()
    catboost_model.fit(x_train.cpu().numpy(), y_train.cpu().numpy(), verbose=False)
    models.append(catboost_model)
    # 2 . - Train LGBM        
    print("[LGBM]")    
    lgb_model = LGBMClassifier ()
    lgb_model. fit(x_train.cpu().numpy(), y_train.cpu().numpy())
    models.append(lgb_model)
    # 3. - Train MLP
    print("[MLP]")         
    mlp_model = MLPClassifier()
    mlp_model.fit(x_train.cpu().numpy(), y_train.cpu().numpy())
    models.append(mlp_model)
    # 4. - Train RF   
    print("[RF]")      
    rf_model = RandomForestClassifier()
    rf_model.fit(x_train.cpu().numpy(), y_train.cpu().numpy())
    models.append(rf_model)
    # 5. - Train  XGB   
    print("[XGB]")      
    xgb_model = XGBClassifier()
    xgb_model.fit(x_train.cpu().numpy(), y_train.cpu().numpy())
    models.append(xgb_model)
    return models

def initializeResultsTransfer(epsilons, models, x_test, y_test):
    x_test = x_test.cpu().numpy()
    y_test = y_test.cpu().numpy()
    metrics = ["Accuracy", "F1 Score", "Recall", "EIR"]
    attacks = ["MIFGSM"]
    models_names = ["CATBOOST", "LGBM", "MLP", "RF", "XGB"]
    multi_index = pd.MultiIndex.from_product([models_names, attacks, metrics], names=["Models", "Attack", "Metric"])
    df_results = pd.DataFrame(index=multi_index, columns=epsilons)
    for idx, model_name in enumerate(models_names):
        y_pred = models[idx].predict(x_test)
        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        for attack in attacks:
            df_results.loc[(model_name, attack, "Accuracy"), 0.0] = accuracy
            df_results.loc[(model_name, attack, "F1 Score"), 0.0] = f1
            df_results.loc[(model_name, attack, "Recall"), 0.0] = recall
            df_results.loc[(model_name, attack, "EIR"), 0.0] = "-"      

        print(recall)

    return df_results

def test_adversarial_samples_transfer(adv_dir, x_test, y_test, ads_model, anomalous_indices, device):
    adversarial_x_data, adversarial_y_data, _ = read_adv_data(adv_dir)
    X_modified, Y_modified  = x_test.clone().cpu().numpy(), y_test.clone().cpu().numpy()
    X_modified[anomalous_indices] = adversarial_x_data
    Y_modified[anomalous_indices] = adversarial_y_data

    y_pred =  ads_model.predict(X_modified)

    accuracy = accuracy_score(Y_modified, y_pred)
    recall = recall_score(Y_modified, y_pred)
    f1 = f1_score(Y_modified, y_pred)
    print(recall)
    return  accuracy, f1, recall

def obtain_adv_results_transfer(df_results, original_results,  file, stage,  x_test, y_test, anomalous_indices, eps, ads_model, model_name, device):
    # MIFGSM
    adv_dir = f'./data/aexamples/{file}/mifgsm/{stage}/adversarialexamples_eps{str(eps)}.csv'
    accuracy, f1, recall = test_adversarial_samples_transfer(adv_dir, x_test, y_test, ads_model, anomalous_indices, device)
    df_results.loc[(model_name, "MIFGSM", "Accuracy"), eps] = accuracy
    df_results.loc[(model_name,"MIFGSM", "F1 Score"), eps] = f1
    df_results.loc[(model_name,"MIFGSM", "Recall"), eps] = recall
    eir = 1 - (df_results.loc[(model_name,"MIFGSM", "Recall"), eps] / df_results.loc[(model_name,"MIFGSM", "Recall"), 0.0])
    df_results.loc[(model_name,"MIFGSM", "EIR"), eps] = eir * 100
    print(f'Recall: {recall}')
    return df_results
def testTransferability(original_results, epsilons):
    transferability_results = {}
    models_names = ["CATBOOST", "LGBM", "MLP", "RF", "XGB"]
    try_epsilon = [0.0, 0.003, 0.021, 0.090, 0.210]
    # Test normal
    index = 0
    for file in original_results:
        try:
            print(f"processing file {file} number {index}")
            transferability_results[file] = {}
            x_train = original_results[file]['x_train']
            y_train = original_results[file]['y_train']
            x_test = original_results[file]['x_test']
            y_test = original_results[file]['y_test']
            models = trainModelsTransfer(x_train, y_train)
            df_results = initializeResultsTransfer(try_epsilon, models, x_test, y_test)
            transferability_results[file]['models'] = models
            transferability_results[file]['transfer_results'] = df_results
            for idx, model_name in enumerate(models_names):
                for eps in try_epsilon[1:]:
                    transferability_results[file]['transfer_results'] = obtain_adv_results_transfer(transferability_results[file]['transfer_results'] , original_results  , file, 'test',  x_test, y_test, original_results[file]['anomalous_indices_test'], eps, models[idx], model_name, device)
        except Exception as e:
            print(f"File {file} failed because {e}")  
        #if index > 3: 
        #    print(f"Finish with idx {index}")
        #    break
        index = index + 1

# df_results = completeTest(original_results, epsilons, device)
    return transferability_results

In [94]:
transferability_results = testTransferability(original_results, epsilons)

processing file Dundee_Clean_With_Anomalies_All number 0
[CATBOOST]
[LGBM]
[LightGBM] [Info] Number of positive: 47129, number of negative: 121282
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000801 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1540
[LightGBM] [Info] Number of data points in the train set: 168411, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.279845 -> initscore=-0.945230
[LightGBM] [Info] Start training from score -0.945230
[MLP]
[RF]
[XGB]
0.7640735175124264
0.6501560513235464
0.7007282395098833
0.7271413709397757
0.6968558548144723
0.799040573344122
Recall: 0.799040573344122
0.9115131198705352
Recall: 0.9115131198705352
0.9909259045197087
Recall: 0.9909259045197087
0.995954224945093
Recall: 0.995954224945093
0.7508958501907294
Recall: 0.7508958501907294
0.8668362039070627
Recall: 0

In [ ]:
import pickle
with open("transferability_results.pkl", "wb") as f:
    pickle.dump(transferability_results, f)

In [33]:
import pickle
with open("transferability_results.pkl", "rb") as f:
    transferability_results = pickle.load(f)

In [114]:
transferability_results['Dundee_Clean_With_Anomalies']['transfer_results']

0.000      0.003       0.021      0.090  \
Models   Attack Metric                                                 
CATBOOST MIFGSM Accuracy  0.886375   0.918592    0.944516   0.947173   
                F1 Score  0.780176   0.851744    0.903511   0.908554   
                Recall    0.666975   0.773543    0.859297   0.868085   
                EIR              - -15.977809  -28.834951 -30.152566   
LGBM     MIFGSM Accuracy  0.858661   0.883215    0.931064   0.927037   
                F1 Score  0.709106   0.771203    0.876522   0.868359   
                Recall    0.569843   0.651064    0.809343   0.796022   
                EIR              - -14.253247  -42.029221 -39.691558   
MLP      MIFGSM Accuracy  0.806617   0.878517    0.958387   0.966777   
                F1 Score  0.585755   0.774502    0.932731   0.947012   
                Recall    0.452266   0.690102    0.954302   0.982054   
                EIR              - -52.587441 -111.004295 -117.14052   
RF       MIFGSM Accuracy  0.837715   0.820991    0.878041   0.878657   
                F1 Score  0.671497   0.624949    0.771759   0.773172   
                Recall    0.548659    0.49334    0.682054   0.684089   
                EIR              -  10.082617  -24.312932 -24.683864   
XGB      MIFGSM Accuracy  0.859025   0.895632    0.925275    0.92192   
                F1 Score  0.710803   0.800854    0.865051    0.85813   
                Recall     0.57308   0.694172    0.792229   0.781129   
                EIR              - -21.129944  -38.240517 -36.303471   

                               0.210  
Models   Attack Metric                
CATBOOST MIFGSM Accuracy    0.937524  
                F1 Score    0.890016  
                Recall       0.83617  
                EIR       -25.367545  
LGBM     MIFGSM Accuracy    0.890878  
                F1 Score    0.789377  
                Recall      0.676411  
                EIR       -18.701299  
MLP      MIFGSM Accuracy    0.963057  
                F1 Score    0.940728  
                Recall       0.96975  
                EIR      -114.420127  
RF       MIFGSM Accuracy     0.84602  
                F1 Score    0.693464  
                Recall      0.576133  
                EIR        -5.007587  
XGB      MIFGSM Accuracy    0.902064  
                F1 Score    0.815393  
                Recall      0.715449  
                EIR       -24.842615

In [116]:
transferability_results['PaloAlto_2018_2022_Clean_With_Anomalies']['transfer_results']

0.000      0.003      0.021      0.090      0.210
Models   Attack Metric                                                        
CATBOOST MIFGSM Accuracy   0.98189   0.961291    0.95953   0.952944   0.938778
                F1 Score  0.940089    0.86258   0.855423   0.827847   0.763776
                Recall    0.920844   0.787345   0.775931   0.733251   0.641439
                EIR              -   14.49744  15.736998  20.371867  30.342226
LGBM     MIFGSM Accuracy  0.977678    0.95976   0.958802   0.951872   0.934528
                F1 Score  0.924997   0.856126   0.852198   0.822933   0.742702
                Recall     0.89206   0.775931   0.769727   0.724814   0.612407
                EIR              -  13.018081  13.713491  18.748261  31.349096
MLP      MIFGSM Accuracy  0.980703   0.942721   0.939084   0.913853   0.894134
                F1 Score  0.935813   0.781924   0.764819    0.63151   0.505455
                Recall    0.911663   0.665509   0.641935   0.478412    0.35062
                EIR              -  27.000544  29.586282  47.523136  61.540555
RF       MIFGSM Accuracy  0.980741   0.956965   0.955012   0.948235   0.925875
                F1 Score  0.936078   0.844923   0.836738   0.807407   0.699192
                Recall    0.913896   0.759801   0.747146   0.703226   0.558313
                EIR              -  16.861254  18.245995   23.05186  38.908499
XGB      MIFGSM Accuracy  0.980014   0.962555   0.960028   0.953595   0.942836
                F1 Score  0.933452   0.867623   0.857416   0.830584   0.782773
                Recall    0.908437   0.795285   0.778908   0.737221   0.667494
                EIR              -  12.455613  14.258399  18.847309  26.522808

In [117]:
transferability_results['Netherlands_Clean_With_Anomalies']['transfer_results']

0.000      0.003      0.021      0.090      0.210
Models   Attack Metric                                                        
CATBOOST MIFGSM Accuracy   0.93654   0.868634   0.863783   0.854487   0.842361
                F1 Score  0.827283   0.561404   0.537723   0.490085   0.423077
                Recall    0.721689   0.399232     0.3762   0.332054   0.274472
                EIR              -  44.680851   47.87234  53.989362  61.968085
LGBM     MIFGSM Accuracy  0.932498   0.864592   0.863379   0.856508   0.843573
                F1 Score   0.81465   0.542974   0.536986   0.502104   0.431718
                Recall    0.704415   0.381958     0.3762    0.34357    0.28215
                EIR              -  45.776567  46.594005  51.226158  59.945504
MLP      MIFGSM Accuracy   0.93169   0.942199   0.938561   0.923201   0.891673
                F1 Score  0.812846   0.846071   0.834783    0.78458   0.666667
                Recall    0.704415   0.754319   0.737044   0.664107   0.514395
                EIR              -  -7.084469  -4.632153   5.722071  26.975477
RF       MIFGSM Accuracy  0.926031   0.849232   0.848424   0.843977   0.834276
                F1 Score   0.79779   0.478322   0.474053   0.450142    0.39528
                Recall    0.692898   0.328215   0.324376   0.303263   0.257198
                EIR              -  52.631579  53.185596  56.232687  62.880886
XGB      MIFGSM Accuracy  0.937753   0.876718   0.875909   0.869846   0.842765
                F1 Score  0.833333   0.605433   0.601816   0.574074   0.435414
                Recall    0.738964   0.449136   0.445298   0.416507   0.287908
                EIR              -  39.220779   39.74026  43.636364  61.038961

In [118]:
transferability_results['Boulder_Clean_With_Anomalies']['transfer_results']

0.000      0.003      0.021      0.090      0.210
Models   Attack Metric                                                        
CATBOOST MIFGSM Accuracy   0.99506   0.978422   0.974912   0.960614   0.932276
                F1 Score  0.983435   0.923361   0.909771   0.850665   0.712314
                Recall     0.96824   0.858369   0.835193   0.740773   0.553648
                EIR              -  11.347518  13.741135  23.492908  42.819149
LGBM     MIFGSM Accuracy  0.991941   0.979722   0.977122   0.964123   0.940465
                F1 Score  0.972663   0.928243   0.918292   0.865628   0.755342
                Recall    0.946781   0.866094   0.848927    0.76309   0.606867
                EIR              -   8.522212  10.335449  19.401632  35.902085
MLP      MIFGSM Accuracy  0.993761   0.989861   0.987131   0.974132   0.954764
                F1 Score  0.978984   0.965395   0.955665   0.906704   0.824597
                Recall    0.959657   0.933906    0.91588   0.830043   0.702146
                EIR              -   2.683363   4.561717  13.506261  26.833631
RF       MIFGSM Accuracy  0.991291   0.975692   0.973222   0.959574   0.935786
                F1 Score  0.970393   0.912739   0.903013   0.845963   0.730937
                Recall    0.942489   0.839485   0.823176   0.733047   0.575966
                EIR              -  10.928962  12.659381  22.222222  38.888889
XGB      MIFGSM Accuracy  0.993891   0.972053   0.968413   0.955414   0.932016
                F1 Score  0.979431   0.898441   0.883676   0.827552    0.71089
                Recall    0.960515   0.816309   0.792275   0.706438   0.551931
                EIR              -  15.013405  17.515639  26.452189   42.53798

In [119]:
transferability_results['Perth_Clean_With_Anomalies']['transfer_results']

0.000      0.003      0.021      0.090      0.210
Models   Attack Metric                                                        
CATBOOST MIFGSM Accuracy  0.993628   0.991716   0.990696   0.986108   0.982476
                F1 Score  0.979407   0.973063   0.969647   0.953989   0.941252
                Recall    0.961196    0.94907   0.942603     0.9135   0.890461
                EIR              -   1.261564   1.934399   4.962153   7.359125
LGBM     MIFGSM Accuracy  0.985471    0.98802   0.986809   0.981903   0.975594
                F1 Score  0.951899   0.960669   0.956522   0.939368   0.916467
                Recall    0.911884   0.928052   0.920372   0.889248   0.849232
                EIR              -   -1.77305  -0.930851    2.48227   6.870567
MLP      MIFGSM Accuracy  0.985662   0.905499   0.903269   0.896132   0.887784
                F1 Score   0.95284   0.577854   0.563542   0.515746   0.455641
                Recall    0.918755   0.410267    0.39612   0.350849   0.297898
                EIR              -  55.345359  56.885174  61.812582  67.575891
RF       MIFGSM Accuracy  0.985344   0.983751   0.982221   0.975977   0.968011
                F1 Score  0.951354   0.945779   0.940372   0.917704   0.887343
                Recall    0.909054   0.898949   0.889248   0.849636   0.799111
                EIR              -   1.111605   2.178746   6.536238  12.094264
XGB      MIFGSM Accuracy  0.991907   0.990633   0.989486   0.985025   0.979418
                F1 Score   0.97369    0.96942   0.965546   0.950201   0.930253
                Recall    0.949879   0.941795   0.934519   0.906225   0.870655
                EIR              -   0.851064   1.617021   4.595745   8.340426

In [11]:
transferability_results['US_Alabama_clean_WithAnomalies']['transfer_results']

0.000     0.003     0.021      0.090      0.210
Models   Attack Metric                                                      
CATBOOST MIFGSM Accuracy  0.917129  0.916486  0.912986   0.900143   0.880314
                F1 Score  0.806769  0.804977   0.79513   0.757561   0.694724
                Recall    0.866858  0.863636  0.846099   0.781747   0.682391
                EIR              -  0.371594  2.394715   9.818332  21.279934
LGBM     MIFGSM Accuracy  0.919771  0.919057  0.915457     0.9034   0.884271
                F1 Score  0.807381  0.805332  0.794898   0.758586   0.696262
                Recall     0.84252  0.838941  0.820902   0.760487   0.664639
                EIR              -  0.424809  2.565845   9.736619  21.112999
MLP      MIFGSM Accuracy  0.936457  0.935686  0.932086   0.920457   0.902571
                F1 Score  0.833807  0.831449  0.820319   0.782873     0.7204
                Recall    0.798712  0.794846  0.776807    0.71854   0.628919
                EIR              -  0.483958  2.742427  10.037641   21.25829
RF       MIFGSM Accuracy  0.892243  0.891114    0.8882   0.877643   0.859686
                F1 Score   0.76681  0.763791  0.755926   0.726576    0.67334
                Recall    0.887759  0.882105  0.867502   0.814603   0.724624
                EIR              -  0.636994   2.28189   8.240606  18.376068
XGB      MIFGSM Accuracy  0.910857  0.910271  0.907143   0.894086     0.8745
                F1 Score  0.794399  0.792768  0.783981   0.745887    0.68405
                Recall    0.862921  0.859986  0.844309   0.778883   0.680744
                EIR              -  0.340108  2.156781   9.738698  21.111572

In [121]:
transferability_results['Germany_clean_WithAnomalies']['transfer_results']

0.000     0.003     0.021      0.090      0.210
Models   Attack Metric                                                      
CATBOOST MIFGSM Accuracy  0.893729    0.8929  0.889014   0.875714   0.857029
                F1 Score  0.704109  0.701112  0.686873   0.635678   0.556619
                Recall    0.633572   0.62942   0.60995   0.543307   0.449678
                EIR              -  0.655293  3.728392  14.246978  29.024969
LGBM     MIFGSM Accuracy  0.894414  0.893543  0.889829   0.877529   0.858414
                F1 Score  0.700247  0.697024  0.683103   0.634803   0.552288
                Recall    0.617967  0.613601  0.594989   0.533357   0.437581
                EIR              -  0.706591   3.71829  13.691648  29.190316
MLP      MIFGSM Accuracy  0.887214  0.886586    0.8835   0.872057   0.854386
                F1 Score  0.679365  0.677001  0.665271   0.619896   0.543427
                Recall    0.598712  0.595562    0.5801   0.522763   0.434216
                EIR              -  0.526064   3.10856  12.685318  27.474892
RF       MIFGSM Accuracy  0.866357  0.865871  0.860371   0.847343   0.829686
                F1 Score  0.672387  0.670804  0.652616   0.607479    0.54125
                Recall    0.687187  0.684753  0.657194   0.591911   0.503436
                EIR              -  0.354167  4.364583  13.864583  26.739583
XGB      MIFGSM Accuracy    0.8926  0.891814    0.8879   0.874657   0.856429
                F1 Score  0.702116  0.699281  0.684973   0.634142   0.557386
                Recall    0.634216  0.630279  0.610666   0.544309   0.452971
                EIR              -  0.620767  3.713318  14.176072  28.577878

In [122]:
transferability_results['Canada1_clean_WithAnomalies']['transfer_results']

0.000     0.003     0.021     0.090      0.210
Models   Attack Metric                                                     
CATBOOST MIFGSM Accuracy  0.996443  0.996543    0.9963  0.996086   0.995843
                F1 Score  0.991143  0.991394  0.990784  0.990245   0.989633
                Recall     0.99728  0.997781  0.996564   0.99549   0.994273
                EIR              - -0.050244  0.071777  0.179443   0.301464
LGBM     MIFGSM Accuracy  0.992214  0.992043  0.991671    0.9899     0.9883
                F1 Score  0.980684   0.98025  0.979309  0.974798   0.970688
                Recall    0.990336  0.989477  0.987616   0.97874   0.970723
                EIR              -  0.086737  0.274666  1.170943   1.980484
MLP      MIFGSM Accuracy  0.969914    0.9693  0.966743  0.956357   0.938643
                F1 Score  0.922259  0.920546  0.913354  0.883134   0.827517
                Recall    0.894202  0.891124  0.878311  0.826271   0.737509
                EIR              -   0.34422  1.777137  7.596862  17.523215
RF       MIFGSM Accuracy  0.994414  0.994486  0.994243  0.993943   0.993529
                F1 Score  0.986164  0.986343  0.985733  0.984978   0.983934
                Recall    0.997423  0.997781  0.996564  0.995061   0.992985
                EIR              - -0.035883   0.08612  0.236831   0.444955
XGB      MIFGSM Accuracy  0.984557    0.9844  0.983386    0.9803   0.974143
                F1 Score  0.962175  0.961775  0.959189  0.951239   0.935009
                Recall     0.98418  0.983393  0.978311  0.962849   0.931997
                EIR              -  0.080006  0.596407   2.16743   5.302204

In [ ]:
transferability_results['vancover_clean_WithAnomalies']['transfer_results']

0.000     0.003     0.021      0.090      0.210
Models   Attack Metric                                                      
CATBOOST MIFGSM Accuracy  0.918629  0.918571  0.916957   0.910171   0.902386
                F1 Score  0.785574  0.785392  0.780202   0.757893    0.73127
                Recall    0.746886    0.7466  0.738511    0.70451   0.665497
                EIR              -  0.038336  1.121334   5.673759  10.897067
LGBM     MIFGSM Accuracy  0.907143  0.906757  0.904814   0.898671   0.888943
                F1 Score  0.741366  0.740012  0.733149    0.71095   0.674155
                Recall    0.666858  0.664925   0.65519   0.624409   0.575662
                EIR              -  0.289824  1.749678   6.365393  13.675397
MLP      MIFGSM Accuracy  0.878629  0.877929  0.875443     0.8663   0.851886
                F1 Score  0.648082  0.645333  0.635478   0.597964   0.534441
                Recall    0.559986  0.556478  0.544023    0.49821   0.425984
                EIR              -  0.626358  2.850569  11.031574  23.929439
RF       MIFGSM Accuracy    0.9148  0.915386  0.914229   0.906843   0.896629
                F1 Score  0.780493  0.782331  0.778695   0.754969   0.720596
                Recall    0.758984  0.761918   0.75612   0.719112   0.667931
                EIR              - -0.386683  0.377252    5.25323  11.996605
XGB      MIFGSM Accuracy    0.8815  0.881014    0.8781   0.867286   0.849043
                F1 Score  0.670663  0.668867  0.657982   0.615989   0.538861
                Recall    0.604581  0.602147  0.587545   0.533357   0.441947
                EIR              -  0.402557  2.817902  11.780725  26.900308

In [124]:
transferability_results['Portugal_clean_WithAnomalies']['transfer_results']

0.000      0.003      0.021      0.090      0.210
Models   Attack Metric                                                        
CATBOOST MIFGSM Accuracy  0.926471     0.8455   0.778014   0.777829   0.777829
                F1 Score  0.803946   0.474617   0.020301   0.018677   0.018677
                Recall    0.755404   0.349678   0.011525   0.010594   0.010594
                EIR              -  53.709846  98.474367  98.597555  98.597555
LGBM     MIFGSM Accuracy    0.9278   0.852729   0.819343   0.783171   0.783171
                F1 Score  0.802594   0.493341   0.297835   0.019382   0.019382
                Recall    0.735433    0.35927   0.191983   0.010737   0.010737
                EIR              -   51.14853   73.89527  98.540004  98.540004
MLP      MIFGSM Accuracy    0.9253   0.790143   0.790143   0.790143   0.790143
                F1 Score  0.783487        0.0        0.0        0.0        0.0
                Recall    0.677237        0.0        0.0        0.0        0.0
                EIR              -      100.0      100.0      100.0      100.0
RF       MIFGSM Accuracy  0.915143   0.826857   0.825629   0.756757      0.821
                F1 Score  0.796254   0.472447   0.466707   0.057564   0.444691
                Recall    0.830852   0.388475   0.382319   0.037223   0.359127
                EIR              -  53.243732  53.984664  95.519945  56.776083
XGB      MIFGSM Accuracy  0.926457   0.811171   0.776157   0.776157   0.776157
                F1 Score  0.804229   0.274772   0.006719   0.006719   0.006719
                Recall    0.756908   0.179241   0.003794   0.003794   0.003794
                EIR              -  76.319274  99.498771  99.498771  99.498771

In [95]:
def process_data_gb(input_data, device):
    data = input_data.dropna(axis=0)
    data = data.loc[:, data.nunique() > 1]
    data.columns = [col.strip().lower() for col in data.columns]
    new_cols = ['timestamp' if 'starttimestamp' in col else col for col in data.columns]
    data.columns = new_cols
    try:
        data['timestamp'] = pd.to_datetime(data['timestamp'], errors='coerce')
        data = data.sort_values(by='timestamp', ascending=True)
    except:
        print("timestamp fallido")
    features = [col for col in data.columns if col not in ['city', 'timestamp', 'anomaly', 'normal/attack',  'startdate', 'weekdaystart', 'yearstart', 'hourstart', 'minutestart', 'enddate', 'endtimestamp', 'weekdayend', 'yearend', 'hourend', 'minuteend', 'class']]
    #print(data)
    if "connectortype" in features:
        data = encode_variable(data, data.columns.get_loc("connectortype"))
        features = ['connectortype', 'durationcharge' , 'durationsession' , 'energy' , 'tariff' ,
                'cost' , 'meanpower', 'maxpower']
    for feature in features:
        data[feature] = pd.to_numeric(data[feature], errors='coerce')
    print(data)
    data.dropna()
    #data = data.sample(frac=1).reset_index(drop=True)
    x_data = data.loc[:, features].select_dtypes(include=[np.number])
    y_data = data.loc[:, ['anomaly']].values.ravel()
    print(y_data)

    size = len(x_data)
    size_init = int(size * 0.8)
    x_train = x_data[:size_init]
    y_train = y_data[:size_init]
    x_test = x_data[size_init:]
    y_test = y_data[size_init:]

    #x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.3, random_state=42, stratify=y_data)

    df = pd.DataFrame(x_test, columns=features)
    df["anomaly"] = y_test
    df = df.sample(frac=1).reset_index(drop=True)
    x_test = df.loc[:, features].values
    y_test = df.loc[:, ['anomaly']].values.ravel()
    x_train = np.array(x_train, dtype=np.float32) 
    x_test = np.array(x_test, dtype=np.float32)
    x_train = torch.tensor(x_train, dtype=torch.float32).to(device)
    y_train = torch.tensor(y_train, dtype=torch.long).to(device)
    x_test = torch.tensor(x_test, dtype=torch.float32).to(device)
    y_test = torch.tensor(y_test, dtype=torch.long).to(device)

    return data, x_train, x_test, y_train, y_test, features
	

In [96]:
def testTransferability_gb():
    transferability_results = {}
    models_names = ["CATBOOST", "LGBM", "MLP", "RF", "XGB"]
    try_epsilon = [0.0, 0.003, 0.021, 0.090, 0.210]
    # Test normal
    data_path = Path('./data')
    files = [f for f in data_path.iterdir() if f.is_file()]
    for file in files:
        file_name = file.with_suffix('').as_posix().split('/')[1]
        print(f"Processing file {file_name}")
        if 'xlsx' in file.name:
            data = pd.read_excel(file)
        else:
            data = pd.read_csv(file)
        data, x_train, x_test, y_train, y_test, features = process_data_gb(data, device)
        try:
            print(f"processing file {file_name}")
            transferability_results[file_name] = {}
            models = trainModelsTransfer(x_train, y_train)
            df_results = initializeResultsTransfer(try_epsilon, models, x_test, y_test)
            transferability_results[file_name]['models'] = models
            transferability_results[file_name]['transfer_results'] = df_results
            for idx, model_name in enumerate(models_names):
                for eps in try_epsilon[1:]:
                    transferability_results[file_name]['transfer_results'] = obtain_adv_results_transfer(transferability_results[file_name]['transfer_results'] , original_results  , file_name, 'test',  x_test, y_test, original_results[file_name]['anomalous_indices_test'], eps, models[idx], model_name, device)
        except Exception as e:
            print(f"File {file} failed because {e}")  

    return transferability_results

In [97]:
transferability_results_gb = testTransferability_gb()

Processing file Dundee_Clean_With_Anomalies_All


        connectortype  durationcharge  durationsession  energy    tariff  \
110571            2.0            0.62             0.62  20.530  0.900000   
110570            2.0            0.52             0.52   9.270  0.910000   
110569            2.0            0.17             0.17   5.110  0.420000   
178831            2.0            0.62             0.62  14.140  1.120000   
110568            2.0            0.62             0.62  14.140  1.120000   
...               ...             ...              ...     ...       ...   
203089            2.0            6.21             1.25  25.450  0.323096   
51800             2.0            1.01             1.02  28.460  0.321504   
196944            2.0            0.26             0.27  46.034  0.364156   
51799             2.0            0.26             0.27   7.940  0.364156   
51798             2.0            0.49             0.50   8.250  0.360824   

           cost  meanpower  maxpower            startdate  \
110571  18.4800     33.113

In [39]:
import pickle
with open("transferability_results_gb.pkl", "wb") as f:
    pickle.dump(transferability_results_gb, f)

In [ ]:
import pickle
with open("transferability_results_gb.pkl", "rb") as f:
    transferability_results_gb = pickle.load(f)

In [41]:
for file in transferability_results_gb:
    print(file)

US_Alabama_clean_WithAnomalies
Dundee_Clean_With_Anomalies_All
US_Alabama_clean_WithAnomalies_a2
Germany_clean_WithAnomalies_a2
Portugal_clean_WithAnomalies_a2
vancover_clean_WithAnomalies_a2
Germany_clean_WithAnomalies
Canada1_clean_WithAnomalies_a2
Netherlands_Clean_With_Anomalies
Canada1_clean_WithAnomalies
Boulder_Clean_With_Anomalies
Perth_Clean_With_Anomalies
PaloAlto_2018_2022_Clean_With_Anomalies
vancover_clean_WithAnomalies
Portugal_clean_WithAnomalies
Dundee_Clean_With_Anomalies


In [43]:
transferability_results_gb['Dundee_Clean_With_Anomalies']['transfer_results']

0.000      0.003      0.021      0.090      0.210
Models   Attack Metric                                                        
CATBOOST MIFGSM Accuracy  0.886347   0.670284   0.677862   0.678254   0.675289
                F1 Score  0.780087   0.536775   0.552191   0.552978   0.546994
                Recall     0.66679   0.373218   0.388024   0.388789   0.382997
                EIR              -   44.02773  41.807193  41.692479  42.561029
LGBM     MIFGSM Accuracy  0.855557   0.875217   0.881509   0.882012   0.881649
                F1 Score  0.700389   0.863522    0.87129   0.871907   0.871462
                Recall    0.558464    0.77124   0.783533   0.784516   0.783806
                EIR              - -38.100067 -40.301293 -40.477391 -40.350209
MLP      MIFGSM Accuracy  0.834499   0.849348   0.870491   0.860619    0.85936
                F1 Score  0.663138   0.834424   0.860893   0.848685    0.84711
                Recall    0.538853   0.741627   0.782932   0.763645   0.761187
                EIR              - -37.630703 -45.296006 -41.716837 -41.260569
RF       MIFGSM Accuracy  0.836372    0.61782    0.61796   0.618211   0.611779
                F1 Score    0.6675   0.435429   0.435752   0.436334   0.421342
                Recall    0.543293   0.287931   0.288204   0.288696    0.27613
                EIR              -  47.002665  46.952383  46.861875  49.174852
XGB      MIFGSM Accuracy  0.859025   0.872588   0.880362   0.881565   0.886347
                F1 Score  0.710803   0.860408   0.870033   0.871507   0.877332
                Recall     0.57308   0.767142   0.782331    0.78468   0.794023
                EIR              - -33.862873 -36.513246 -36.923195  -38.55346

In [46]:
transferability_results_gb['PaloAlto_2018_2022_Clean_With_Anomalies']['transfer_results']

0.000      0.003      0.021     0.090      0.210
Models   Attack Metric                                                       
CATBOOST MIFGSM Accuracy   0.98189   0.907918   0.903323  0.935677   0.915729
                F1 Score  0.940089   0.810883   0.799555  0.875019   0.829657
                Recall    0.920844   0.694504   0.678341  0.792161   0.721983
                EIR              -  24.579564  26.334889  13.97448  21.595513
LGBM     MIFGSM Accuracy  0.977142   0.962631   0.963397  0.962976   0.966613
                F1 Score  0.923294   0.931034   0.932543  0.931714   0.938833
                Recall    0.891563   0.887392   0.890086  0.888605   0.901401
                EIR              -   0.467834   0.165672  0.331861  -1.103409
MLP      MIFGSM Accuracy  0.982273   0.984187   0.984225  0.983575   0.982962
                F1 Score    0.9414   0.971907   0.971977  0.970787   0.969664
                Recall    0.922829   0.962284   0.962419  0.960129   0.957974
                EIR              -  -4.275517  -4.290113 -4.041977  -3.808437
RF       MIFGSM Accuracy  0.980435   0.965579   0.965196  0.964928      0.966
                F1 Score  0.934996    0.93681   0.936062  0.935538   0.937632
                Recall    0.911911   0.897629   0.896282  0.895339   0.899111
                EIR              -   1.566092   1.713802  1.817199   1.403611
XGB      MIFGSM Accuracy  0.980014   0.969446   0.968489   0.97228   0.970021
                F1 Score  0.933452   0.944282   0.942435  0.949708   0.945386
                Recall    0.908437    0.91083   0.907462  0.920797    0.91285
                EIR              -  -0.263421   0.107266 -1.360655  -0.485834

In [47]:
transferability_results_gb['Perth_Clean_With_Anomalies']['transfer_results']

0.000     0.003     0.021     0.090      0.210
Models   Attack Metric                                                     
CATBOOST MIFGSM Accuracy  0.993628  0.990123  0.989677  0.988339   0.985089
                F1 Score  0.979407   0.98268  0.981883  0.979487   0.973619
                Recall    0.961196  0.966799  0.965259  0.960642   0.949428
                EIR              - -0.582831 -0.422704  0.057678    1.22432
LGBM     MIFGSM Accuracy   0.98528   0.97929  0.978972  0.978271    0.97604
                F1 Score  0.951399  0.963098  0.962509   0.96121   0.957058
                Recall    0.913905  0.932498  0.931398   0.92898   0.921284
                EIR              - -2.034479 -1.914183 -1.649534  -0.807466
MLP      MIFGSM Accuracy   0.99229  0.973874  0.976486  0.969477   0.957561
                F1 Score  0.975128  0.952992  0.957891  0.944643   0.921332
                Recall    0.958771  0.913808  0.922823  0.898637    0.85752
                EIR              -  4.689644  3.749383  6.272034  10.560541
RF       MIFGSM Accuracy  0.985854  0.973555  0.974065  0.972344   0.971325
                F1 Score  0.953085  0.952227  0.953191  0.949931   0.947989
                Recall    0.911479  0.909411   0.91117  0.905233   0.901715
                EIR              -  0.226956  0.033971  0.685294   1.071264
XGB      MIFGSM Accuracy  0.991907  0.986491  0.986236  0.985535   0.981393
                F1 Score   0.97369  0.976153  0.975692  0.974423   0.966856
                Recall    0.949879  0.954046  0.953166  0.950748   0.936456
                EIR              - -0.438687 -0.346096 -0.091469   1.413144

In [48]:
transferability_results_gb['Boulder_Clean_With_Anomalies']['transfer_results']

0.000     0.003     0.021     0.090      0.210
Models   Attack Metric                                                     
CATBOOST MIFGSM Accuracy   0.99506  0.990771  0.989861  0.987781   0.978032
                F1 Score  0.983435  0.983322  0.981647  0.977799   0.959365
                Recall     0.96824  0.967638  0.964401  0.957004    0.92233
                EIR              -  0.062258  0.396498  1.160475   4.741617
LGBM     MIFGSM Accuracy  0.992201  0.989861  0.989341  0.987001   0.985961
                F1 Score  0.973568  0.981638  0.980679  0.976337   0.974395
                Recall    0.948498  0.963939   0.96209  0.953768   0.950069
                EIR              - -1.627955 -1.432985 -0.555622  -0.165682
MLP      MIFGSM Accuracy   0.99766  0.981542  0.979072  0.970623   0.956324
                F1 Score  0.992215  0.966061  0.961345  0.944878   0.915789
                Recall    0.984549   0.93435  0.925566  0.895515    0.84466
                EIR              -  5.098669  5.990864  9.043109  14.208446
RF       MIFGSM Accuracy  0.990121  0.980372  0.979982  0.976472   0.963343
                F1 Score  0.966282  0.963832  0.963087  0.956333   0.930267
                Recall    0.934764   0.93019  0.928803   0.91632   0.869626
                EIR              -  0.489364   0.63774  1.973121   6.968436
XGB      MIFGSM Accuracy  0.993891  0.990121  0.989731  0.988821   0.977382
                F1 Score  0.979431  0.982126  0.981407  0.979727   0.958113
                Recall    0.960515  0.965326  0.963939  0.960703   0.920018
                EIR              - -0.500868  -0.35647 -0.019542   4.216127

In [49]:
transferability_results_gb['Netherlands_Clean_With_Anomalies']['transfer_results']

0.000      0.003      0.021      0.090      0.210
Models   Attack Metric                                                        
CATBOOST MIFGSM Accuracy   0.93654   0.941795   0.941795   0.938965   0.912692
                F1 Score  0.827283   0.917336   0.917336   0.912968   0.870659
                Recall    0.721689   0.856377   0.856377   0.848875   0.779207
                EIR              - -18.662915 -18.662915 -17.623315  -7.969887
LGBM     MIFGSM Accuracy  0.934519    0.93654   0.934923   0.936136   0.928456
                F1 Score  0.821978   0.909406   0.906883   0.908776   0.896673
                Recall     0.71785   0.844587     0.8403   0.843516   0.823151
                EIR              - -17.655083 -17.057849 -17.505775 -14.668913
MLP      MIFGSM Accuracy  0.931285    0.93169   0.930073   0.927243   0.920372
                F1 Score   0.81441   0.902706   0.900173   0.895713   0.884728
                Recall    0.715931     0.8403   0.836013    0.82851   0.810289
                EIR              - -17.371677 -16.772842 -15.724881 -13.179832
RF       MIFGSM Accuracy  0.928456   0.937348   0.936136   0.934115   0.915117
                F1 Score  0.803987   0.910971   0.909091   0.905943   0.875445
                Recall    0.696737   0.849946   0.846731   0.841372   0.790997
                EIR              - -21.989554 -21.528055  -20.75889  -13.52874
XGB      MIFGSM Accuracy  0.937753   0.945837   0.944624   0.942603   0.902183
                F1 Score  0.833333   0.923777   0.921937   0.918857   0.853333
                Recall    0.738964   0.870311   0.867095   0.861736   0.754555
                EIR              -  -17.77453 -17.339402  -16.61419  -2.109937

In [50]:
transferability_results_gb['vancover_clean_WithAnomalies']['transfer_results']

0.000      0.003      0.021      0.090      0.210
Models   Attack Metric                                                        
CATBOOST MIFGSM Accuracy  0.918914   0.935286   0.935286   0.935286   0.935286
                F1 Score  0.786633   0.907871   0.907871   0.907871   0.907871
                Recall    0.748962     0.8885     0.8885     0.8885     0.8885
                EIR              - -18.630797 -18.630797 -18.630797 -18.630797
LGBM     MIFGSM Accuracy  0.900314   0.920014   0.920014   0.920014   0.920014
                F1 Score  0.722985   0.883507   0.883507   0.883507   0.883507
                Recall    0.651825   0.845189   0.845189   0.845189   0.845189
                EIR              - -29.664993 -29.664993 -29.664993 -29.664993
MLP      MIFGSM Accuracy  0.883029   0.906657   0.906657   0.906657   0.906657
                F1 Score  0.635311   0.857597   0.857597   0.857597   0.857597
                Recall    0.510523   0.783209   0.783209   0.783209   0.783209
                EIR              - -53.413257 -53.413257 -53.413257 -53.413257
RF       MIFGSM Accuracy  0.917414   0.934071   0.934071   0.934071   0.934071
                F1 Score  0.787017   0.907081   0.907081   0.907081   0.907081
                Recall    0.764567     0.8967     0.8967     0.8967     0.8967
                EIR              - -17.282077 -17.282077 -17.282077 -17.282077
XGB      MIFGSM Accuracy    0.8815     0.9056     0.9056     0.9056     0.9056
                F1 Score  0.670663   0.862585   0.862585   0.862585   0.862585
                Recall    0.604581   0.825604   0.825604   0.825604   0.825604
                EIR              - -36.558003 -36.558003 -36.558003 -36.558003

In [51]:
transferability_results_gb['US_Alabama_clean_WithAnomalies']['transfer_results']

0.000     0.003     0.021     0.090     0.210
Models   Attack Metric                                                    
CATBOOST MIFGSM Accuracy  0.917129  0.924829  0.924943  0.924471  0.924671
                F1 Score  0.806769  0.897271  0.897443  0.896732  0.897034
                Recall    0.866858  0.914736  0.915055  0.913741  0.914298
                EIR              -  -5.52323 -5.559966 -5.408431 -5.472718
LGBM     MIFGSM Accuracy  0.921886  0.927214  0.927214  0.927243  0.927957
                F1 Score  0.812316  0.898967  0.898967  0.899011  0.900101
                Recall    0.847029  0.902277  0.902277  0.902357  0.904347
                EIR              - -6.522506 -6.522506 -6.531905 -6.766878
MLP      MIFGSM Accuracy  0.933871  0.934771    0.9348  0.934229  0.934614
                F1 Score  0.828358  0.905898  0.905943  0.905041   0.90565
                Recall    0.799571  0.874851   0.87493  0.873338  0.874413
                EIR              - -9.415082 -9.425039 -9.225903  -9.36032
RF       MIFGSM Accuracy  0.893357  0.907143    0.9072    0.9064  0.906571
                F1 Score  0.768835  0.877714  0.877798  0.876615  0.876869
                Recall    0.888618  0.928549  0.928708  0.926479  0.926956
                EIR              - -4.493516 -4.511434 -4.260582 -4.314336
XGB      MIFGSM Accuracy  0.910857  0.919386  0.919486  0.919114  0.919443
                F1 Score  0.794399    0.8904   0.89055   0.88999  0.890486
                Recall    0.862921  0.912427  0.912706  0.911671  0.912587
                EIR              - -5.737123 -5.769413 -5.649477 -5.755574

In [53]:
transferability_results_gb['Germany_clean_WithAnomalies']['transfer_results']

0.000      0.003      0.021      0.090      0.210
Models   Attack Metric                                                        
CATBOOST MIFGSM Accuracy  0.893529   0.855043   0.854029   0.849914   0.843071
                F1 Score  0.704093   0.768908   0.766915    0.75876   0.744956
                Recall    0.634717   0.670999   0.668177   0.656729    0.63769
                EIR              -  -5.716251  -5.271618  -3.468035  -0.468327
LGBM     MIFGSM Accuracy  0.894871   0.859057   0.858171   0.854057   0.847914
                F1 Score  0.702246   0.774419   0.772679   0.764532   0.752165
                Recall    0.621188   0.673146   0.670681   0.659234   0.642142
                EIR              -  -8.364205  -7.967477  -6.124614  -3.373117
MLP      MIFGSM Accuracy  0.879871   0.750857   0.749743   0.746357   0.741257
                F1 Score  0.648085   0.520088   0.516905   0.507148   0.492206
                Recall    0.554259   0.375626   0.372526   0.363105   0.348915
                EIR              -  32.229164  32.788542  34.488192  37.048424
RF       MIFGSM Accuracy  0.865114   0.823329   0.822457   0.819643      0.814
                F1 Score  0.669675   0.731357   0.729673   0.724206   0.713102
                Recall    0.685111   0.669131   0.666706   0.658876   0.643175
                EIR              -   2.332449   2.686359   3.829314   6.121026
XGB      MIFGSM Accuracy  0.893371   0.837614   0.836671   0.833757   0.832986
                F1 Score  0.703574   0.733713   0.731752   0.725652   0.724028
                Recall    0.634073   0.622466   0.619843   0.611734   0.609587
                EIR              -   1.830546   2.244287   3.523123   3.861638

In [54]:
transferability_results_gb['Canada1_clean_WithAnomalies']['transfer_results']

0.000      0.003      0.021      0.090      0.210
Models   Attack Metric                                                        
CATBOOST MIFGSM Accuracy  0.996443   0.996371   0.996457   0.996386   0.996457
                F1 Score  0.991143   0.994949   0.995069   0.994969   0.995069
                Recall     0.99728   0.996812   0.997052   0.996852   0.997052
                EIR              -   0.046867   0.022895   0.042871   0.022895
LGBM     MIFGSM Accuracy  0.997286   0.995143   0.995143   0.994914   0.994657
                F1 Score  0.993201   0.993202   0.993202    0.99288   0.992517
                Recall    0.993343   0.989601   0.989601   0.988963   0.988246
                EIR              -   0.376719   0.376719   0.440896   0.513096
MLP      MIFGSM Accuracy      0.97   0.923886   0.923457   0.921586   0.917957
                F1 Score  0.923736   0.884865   0.884141   0.880971   0.874774
                Recall    0.910379   0.815762   0.814567   0.809347   0.799227
                EIR              -  10.393158  10.524457  11.097794  12.209454
RF       MIFGSM Accuracy  0.994114   0.994814     0.9948   0.994729   0.994657
                F1 Score  0.985436   0.992804   0.992784   0.992684   0.992584
                Recall    0.997709   0.997729   0.997689    0.99749   0.997291
                EIR              -  -0.001957   0.002036   0.022004   0.041972
XGB      MIFGSM Accuracy  0.984557   0.965571   0.965143   0.964886     0.9646
                F1 Score  0.962175    0.95103    0.95039   0.950006   0.949579
                Recall     0.98418   0.932425    0.93123   0.930512   0.929716
                EIR              -    5.25874   5.380193   5.453065   5.534033

In [55]:
transferability_results_gb['Portugal_clean_WithAnomalies']['transfer_results']

0.000      0.003      0.021      0.090      0.210
Models   Attack Metric                                                        
CATBOOST MIFGSM Accuracy  0.926786   0.901843   0.874114   0.876071   0.940643
                F1 Score  0.804307   0.850783   0.800209   0.803924   0.914792
                Recall    0.753901   0.779746    0.70248   0.707934   0.887863
                EIR              -  -3.428143   6.820684     6.0973 -17.769109
LGBM     MIFGSM Accuracy  0.926343   0.900971     0.9009   0.875514   0.940157
                F1 Score  0.798877   0.847929   0.847803   0.801059    0.91332
                Recall    0.732999   0.769317   0.769117    0.69838   0.878508
                EIR              -  -4.954606  -4.927452   4.722985 -19.851143
MLP      MIFGSM Accuracy  0.918014     0.8949   0.860357   0.864186   0.933457
                F1 Score  0.752384   0.831559   0.763082   0.771065    0.89956
                Recall    0.624123   0.722901   0.626647   0.637315   0.830341
                EIR              - -15.826691  -0.404393  -2.113729 -33.041242
RF       MIFGSM Accuracy  0.915186     0.9304     0.9313   0.930243   0.931886
                F1 Score  0.796713   0.904695   0.906043   0.904459   0.906919
                Recall    0.832785   0.920505   0.923013   0.920067   0.924645
                EIR              - -10.533363 -10.834505 -10.480783 -11.030486
XGB      MIFGSM Accuracy  0.926457   0.901729   0.901871   0.873886   0.940486
                F1 Score  0.804229   0.850875   0.851124   0.800181   0.914705
                Recall    0.756908   0.781219   0.781617   0.703634   0.889216
                EIR              -  -3.211916  -3.264508   7.038276 -17.480142